# 🗺️ Feature Maps — Notes + Interview
---
> **Simple English** | **Interview Ready**

## 📌 What is a Feature Map? (Simple English)
- Feature Map = the **output** produced by applying one filter to the input
- It's a 2D grid showing **where that filter's pattern was found** in the image
- High value in feature map = filter found a strong match at that location
- Low/zero value = pattern not present there
- 32 filters → 32 feature maps stacked together = output volume
- Also called: **activation map**

## 🔑 Feature Map Dimensions
```
Input:   H × W × C   (height × width × channels)
Filter:  f × f × C   (same depth as input)
Output:  H' × W' × N  (N = number of filters)

Where H' = (H - f + 2P)/S + 1
      W' = (W - f + 2P)/S + 1
```

## 🧱 What Each Layer's Feature Maps Represent
| Layer | Feature Map Contains |
|---|---|
| Layer 1 | Edges, colors, corners |
| Layer 2 | Textures, simple shapes |
| Layer 3 | Object parts (eyes, wheels) |
| Deep layers | High-level semantics (faces, cars) |

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Build a small CNN and visualize feature maps
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_sample = x_train[:1].reshape(1,28,28,1).astype('float32') / 255.0

# Model with 8 filters so we can visualize
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(8, (3,3), activation='relu', padding='same', input_shape=(28,28,1)),
    tf.keras.layers.Conv2D(16,(3,3), activation='relu', padding='same'),
])
model.build()

# Feature maps from layer 1
layer1_output = tf.keras.Model(inputs=model.input,
                               outputs=model.layers[0].output)
fmaps = layer1_output.predict(x_sample, verbose=0).squeeze()  # shape: 28×28×8

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes[0,0].imshow(x_sample.squeeze(), cmap='gray')
axes[0,0].set_title('Original Input')
axes[0,0].axis('off')
for i in range(8):
    r,c = (i+1)//5, (i+1)%5
    axes[r,c].imshow(fmaps[:,:,i], cmap='viridis')
    axes[r,c].set_title(f'Filter {i+1} map')
    axes[r,c].axis('off')
axes[1,4].axis('off')
plt.suptitle('Feature Maps from Layer 1 — Each filter detects different patterns', fontsize=11)
plt.tight_layout(); plt.show()
print(f"Input shape: {x_sample.shape}")
print(f"Feature map shape: {fmaps.shape}  (28×28 × 8 filters)")

In [ ]:
# Show how depth grows through CNN layers
model_deep = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(28,28,1)),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(128,(3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(2,2),
])
model_deep.build()
print("Volume transformations through layers:")
print(f"Input        : 28 × 28 × 1")
shapes = [l.output_shape for l in model_deep.layers]
names  = ['Conv1 (32 filters)','Conv2 (64 filters)','MaxPool','Conv3 (128 filters)','MaxPool']
for name, shape in zip(names, shapes):
    print(f"{name:25s}: {shape[1]} × {shape[2]} × {shape[3]}")
print("\n→ Width/Height shrinks | Depth (channels) grows as we go deeper!")

## 🗣️ Interview Q&A

**Q: What is a feature map?**
> The output of applying a convolutional filter to the input. It's a 2D activation map showing where that filter's pattern was detected. One filter = one feature map.

**Q: Why do we have many feature maps?**
> Each filter detects a different pattern (edges, curves, textures). More filters = more patterns detected = richer representation.

**Q: What happens to feature map size as network gets deeper?**
> Width/height shrinks (due to convolutions and pooling). Depth (number of channels/feature maps) grows. This creates a funnel shape.

**Q: How to visualize feature maps?**
> Create intermediate model: `tf.keras.Model(inputs=model.input, outputs=model.layers[i].output)` then predict on sample image and plot each channel.